In [1]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100   2134 100   2134   0      0   5248      0                              0
100   2134 100   2134   0      0   5247      0                              0
100   2134 100   2134   0      0   5247      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100    888 100    888   0      0   2528      0                              0
100    888 100    888   0      0   2527      0                              0
100    888 100    888   0      0   2527      0                              0


In [3]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import os

# uncomment below to use openai
# openai_client = OpenAI()

# uncomment below if using groq instead
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    model='qwen/qwen3-32b')   # if using openai instead of groq, no need to pass model parameter

In [6]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps:

### 1. **Install Ollama**
Download and install based on your operating system:
- **macOS/Windows**: Visit [https://ollama.com/download](https://ollama.com/download) and install the `.pkg` (macOS) or `.msi` (Windows) file.
- **Linux**:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

### 2. **Start the Ollama Server**
After installation, launch the local server:
```bash
ollama serve
```
This runs the server on `localhost:11434`. For persistent background execution (e.g., in Jupyter notebooks):
```bash
nohup ollama serve > nohup.out 2>&1 &
```

### 3. **Run a Model**
Download and run a model (e.g., LLaMA 3):
```bash
ollama run llama3
```
This triggers:
- Model download (~4GB).
- Starts the model for interaction.

### 4. **Test the Server**
Verify the server is running:
```bash
curl http://localhost:11434
```
Expected output:
```json
{"models": [...]}  
```

### 5. **Use the Python Client**
Install the Ollama Python client:
```ba

In [ ]:
# ideally, should be replying "I don't know" but llm internally has the information and hence replies
answer = assistant.rag('How do I run Olama locally?')
print(answer)

To run **Olama locally**, follow these steps based on the course's local setup guidelines:

---

### 1. **Environment Setup**
- **Python**: Install Python 3.10+ and use a tool like `uv` (a fast, modern Python dependency manager) to set up a virtual environment:
  ```bash
  uv venv .
  source venv/bin/activate  # On Windows: venv\Scripts\activate
  uv sync  # Installs dependencies from `requirements.txt` if available
  ```

- **Dependencies**: Install required libraries (e.g., `fastapi`, `jupyter`, `docker`, etc.) using `uv`:
  ```bash
  uv pip install package_name
  ```

- **Docker**: If the tool requires containers:
  ```bash
  docker-compose up
  ```

---

### 2. **API Key/Secret Management**
- Use **`dirdotenv`** to manage keys in `.env` files:
  ```bash
  uv tool install dirdotenv
  eval "$(dirdotenv hook bash)"  # Add to your shell config (e.g., `.bashrc`)
  ```
  - In your project directory, create a `.env` file:
    ```bash
    echo "API_KEY=your_token_here" > .env
    echo ".en

In [ ]:
# a generic reply that has nothing to do with course user wants to know about
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='qwen/qwen3-32b',
    input=messages,
)

response.output_text

'Of course! You can join the course, but I’d recommend confirming a few details to ensure it works for your timing and needs. Here’s what to consider:\n\n### 1. **Enrollment Window**  \n   - **Ongoing Enrollment:** If the course is open for enrollment (many are open year-round), you can join now and start at your own pace.  \n   - **Deadline Check:** If there’s a registration deadline or session-specific timing (e.g., starts/ends on specific dates), let me know which course you’re referring to, and I’ll clarify!  \n\n### 2. **Catch-Up Resources**  \n   If the course is in progress, you’ll typically still have access to:  \n   - Archived lectures or content from earlier modules.  \n   - Support via forums or office hours (depending on the platform).  \n\n### 3. **Prerequisites**  \n   Let me know if you’re unsure about the level (beginner/intermediate/advanced) or required skills/knowledge—it varies by course. I can help you assess if you’re ready.  \n\n### 4. **Access to Materials**  \

In [9]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [11]:
response = openai_client.responses.create(
    model='qwen/qwen3-32b',
    input=messages,
    tools=[search_tool]
)

In [ ]:
# length is 1 in case of openai model but groq does its own reasoning first
# we will assign the function call to index 1 to point to correct function call info
len(response.output)

2

In [13]:
response.output

[ResponseReasoningItem(id='resp_01kvrft1bbf9frah9m1dy46e91', summary=[], type='reasoning', content=[Content(text='Okay, the user is asking if they can join the course since they just discovered it. I need to check the FAQ to see if there\'s information about joining late or enrollment deadlines.\n\nFirst, I\'ll use the search function to look up entries related to joining the course. The query should be something like "Can I join the course late?" or "Enrollment deadline". Let me structure the tool call with the query parameter set to "joining the course late" to find relevant FAQ entries.\n\nI\'ll make sure the JSON is correctly formatted with the function name as "search" and the arguments including the query. That should retrieve any existing FAQ answers on this topic.\n', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"joining the course late"}', call_id='mq8nxe7ca', name='search', type='function_call', id='mq8nxe

In [15]:
call = response.output[1]

In [16]:
import json

args = json.loads(call.arguments)
args

{'query': 'joining the course late'}

In [17]:
call.name

'search'

In [18]:
results = search(**args)

In [19]:
result_json = json.dumps(results, indent=2)

In [20]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [23]:
function_call_output

{'type': 'function_call_output',
 'call_id': 'mq8nxe7ca',
 'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\\n\\nA typical workflow is:\\n\\n1. Watch the lesson videos.\\n2. Work through the lesson notebooks/code.\\n3. Read the homework instructions on GitHub.\\n4. Submit answers through the course platform before the deadline.\\n\\nHomework is similar to the lesson flow, b

In [24]:
# messages.append(call)
messages.extend(response.output)

In [25]:
messages.append(function_call_output)

In [26]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kvrft1bbf9frah9m1dy46e91', summary=[], type='reasoning', content=[Content(text='Okay, the user is asking if they can join the course since they just discovered it. I need to check the FAQ to see if there\'s information about joining late or enrollment deadlines.\n\nFirst, I\'ll use the search function to look up entries related to joining the course. The query should be something like "Can I join the course late?" or "Enrollment deadline". Let me structure the tool call with the query parameter set to "joining the course late" to find relevant FAQ entries.\n\nI\'ll make sure the JSON is correctly formatted with the function name as "search" and the arguments including the query. That should retrieve any existing FAQ answers on this topic.\n', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"joining the course

In [27]:
response = openai_client.responses.create(
    model='qwen/qwen3-32b',
    input=messages,
    tools=[search_tool]
)

In [ ]:
# this time its a relevant response
print(response.output_text)

You can join the course at any time! Here's what you need to know:

1. **Immediate Access**: The course materials (videos, GitHub repo, and lesson notebooks) are all available now. You can start learning at your own pace.

2. **Certificate Eligibility**: 
   - To earn a certificate, you must submit your capstone project while the course is still accepting submissions (deadline dates are listed in the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/)).
   - Certificates are only awarded to those who complete the course during a "live" cohort. Self-paced learners (starting after the cohort) are not eligible for certificates. 

3. **No Registration Needed**: You don’t need to wait for a confirmation email to start. Simply begin working through the materials and submitting homework while the submission window is open.

If you're ready to start, dive into the [LLM Zoomcamp documentation](https://datatalks.club/docs/courses/llm-zoomcamp/) and follow the workflow suggested 

In [29]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(912, 510)

In [30]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(912, 510)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0004428


In [31]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [50]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [51]:
response = openai_client.responses.create(
    model='qwen/qwen3-32b',
    input=messages,
    tools=[search_tool]
)

In [52]:
response.output

[ResponseReasoningItem(id='resp_01kvrhgt3afht9nqgga4dyy07y', summary=[], type='reasoning', content=[Content(text='Okay, the user is asking if they can join the course since they just discovered it. I need to check the FAQ for any information related to joining deadlines or policies.\n\nFirst, I\'ll use the search function to look up entries that mention "join" or "deadline". The function requires a query parameter, so I\'ll input "join course" as the search term. That should retrieve any relevant FAQ entries about joining procedures or time limits. Once I get the results, I can look through them to find the answer. If there\'s a specific deadline mentioned, I\'ll let the user know. If not, I\'ll explain the general process for joining. If the search doesn\'t return enough info, I might need to ask for more details or direct them to contact support. But for now, proceed with the tool call.\n', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCal

In [53]:
# openai did not have reasoning in the response but groq does, so we are adding it here to see in the printed output along with call output
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('FUNCTION CALL:', item.name, item.arguments)
        call_output = make_call(item)
        print('FUNCTION OUTPUT:', call_output['output'])
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)
    
    elif item.type == 'reasoning':
        print('REASONING:')
        print(item.content[0].text)

REASONING:
Okay, the user is asking if they can join the course since they just discovered it. I need to check the FAQ for any information related to joining deadlines or policies.

First, I'll use the search function to look up entries that mention "join" or "deadline". The function requires a query parameter, so I'll input "join course" as the search term. That should retrieve any relevant FAQ entries about joining procedures or time limits. Once I get the results, I can look through them to find the answer. If there's a specific deadline mentioned, I'll let the user know. If not, I'll explain the general process for joining. If the search doesn't return enough info, I might need to ask for more details or direct them to contact support. But for now, proceed with the tool call.

FUNCTION CALL: search {"query":"join course"}
FUNCTION OUTPUT: [
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still

In [54]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kvrhgt3afht9nqgga4dyy07y', summary=[], type='reasoning', content=[Content(text='Okay, the user is asking if they can join the course since they just discovered it. I need to check the FAQ for any information related to joining deadlines or policies.\n\nFirst, I\'ll use the search function to look up entries that mention "join" or "deadline". The function requires a query param

In [ ]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'ITERATION #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='qwen/qwen3-32b',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('FUNCTION CALL:', item.name, item.arguments)
            call_output = make_call(item)
            print('FUNCTION OUTPUT:', call_output['output'])
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
        
        elif item.type == 'reasoning':
            print('REASONING:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

ITERATION #1...
REASONING:
Okay, the user is asking if they can join the course since they just discovered it. I need to check the FAQ to see if there's information about joining after the start date. Let me use the search function to look for relevant entries.

I'll search for terms like "Can I join late?" or "Join after start date". Let me try "Can I join the course late?" as the query. That should find any FAQs addressing late enrollment or joining after the course has started.

FUNCTION CALL: search {"query":"Can I join the course late?"}
FUNCTION OUTPUT: [
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.",
    "doc_id": "74eb249bbf"
  },
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Cert

In [56]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('FUNCTION CALL:', item.name, item.arguments)
                call_output = make_call(item)
                print('FUNCTION OUTPUT:', call_output['output'])
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)
            
            elif item.type == 'reasoning':
                print('REASONING:')
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [57]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [58]:
result = agent_loop(instructions, question, model='qwen/qwen3-32b')

iteration #1...
REASONING:
Okay, the user is asking if they can join the course since they just discovered it. I need to check the FAQ for any information related to joining the course. The available function is "search" which takes a query parameter. I should use that function to look up entries related to joining the course. Let me use the search function with the query "join course" to find relevant FAQ entries.

FUNCTION CALL: search {"query":"join course"}
FUNCTION OUTPUT: [
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions.",
    "doc_id": "74eb249bbf"
  },
  {
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "When will the course be offered next?",
    "answer": "Summer 2027.",
    "doc_id": "bd

In [59]:
result

'You can join the course at any time! However, there are a few important details to keep in mind:\n\n1. **Certificate eligibility**: To receive a certificate, you must submit your project while the course is still accepting submissions (certificates are only awarded to those who finish with a "live" cohort).\n\n2. **No registration required**: You don’t need a confirmation email to start—registration is only for gauging interest. You can begin learning and submitting homework immediately.\n\n3. **Next offering**: If you miss the current submission window, the course will be offered again in **Summer 2027**.\n\nTo get started, follow the workflow outlined in the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/) and use the GitHub repository materials. Let me know if you need further guidance!'

In [ ]:
# here groq responses with information from outside the context and reasoning indicates a lot of confusion
# openai obviously doesn't show such behavior as seen in original notebook, and also responds as intended
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question, model='qwen/qwen3-32b')

iteration #1...
REASONING:
Okay, the user is asking "what's queen gambit?" I need to figure out what they're referring to. The Queen's Gambit is a well-known opening in chess, so maybe they're asking about that. But there's also a popular TV show called "The Queen's Gambit" which is a Netflix series about a female chess prodigy. The user might be referring to either.

Since the tools available include a search function for the FAQ, I should use that to check if there's an entry explaining the Queen's Gambit. The function requires a query parameter. I'll use "Queen's Gambit" as the search term. If the FAQ has an entry, the search will find it. If not, I might need to provide a general explanation. But since the user hasn't specified a context, I'll proceed with the search to see if the FAQ has relevant information.

FUNCTION CALL: search {"query":"Queen's Gambit"}
FUNCTION OUTPUT: []
iteration #2...
REASONING:
Okay, the user asked about the Queen's Gambit. I need to figure out what they

In [ ]:
# this second agent call with same groq model returns as intended
# maybe some temperature parameter or somehting else needs to be taken care of to ensure more determinism
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question, model='qwen/qwen3-32b')

iteration #1...
REASONING:
Okay, the user is asking about "Queen Gambit." I need to figure out what they're referring to. The Queen's Gambit is a chess opening where White offers a pawn to divert the Black e-pawn. But maybe they're thinking of the Netflix show "The Queen's Gambit," which is a fictional story about a chess prodigy. Since there's a possibility of confusion between the actual chess strategy and the movie, I should check the FAQ to see if there's information on either. Let me use the search function to look up both terms to ensure I provide accurate information. I'll search for "Queen Gambit" and "The Queen's Gambit" to cover both bases.

FUNCTION CALL: search {"query":"Queen Gambit"}
FUNCTION OUTPUT: []
iteration #2...
REASONING:
Okay, the user asked about the Queen Gambit, and my initial search using the provided function didn't return any results. Hmm, maybe the FAQ database doesn't have an entry specifically named "Queen Gambit." Let me think. The Queen Gambit is a wel

In [62]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [63]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [64]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [65]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [66]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [67]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [70]:
# uncomment below to use openai
# runner = OpenAIResponsesRunner(
#     tools=agent_tools,
#     developer_prompt=instructions,
#     chat_interface=chat_interface,
#     llm_client=OpenAIClient(model='gpt-5.4-mini')
# )

# uncomment below if using groq instead
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='qwen/qwen3-32b',
                            client=openai_client)
)

In [71]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


c:\Valinor\Programs\miniconda3\envs\llm_zoomcamp\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3-32b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [72]:
result.cost

In [73]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kvrmf8veendrmh1y4fay39cf', summary=[], type='reasoning', content=[Cont

In [74]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


In [ ]:
# this brings up interactive chat and can be stoped by sending 'stop' message as the user input
runner.run();

-> Response received


-> Response received


c:\Valinor\Programs\miniconda3\envs\llm_zoomcamp\Lib\site-packages\toyaikit\chat\runners.py:283: UnknownModelWarning: No pricing data for model 'qwen/qwen3-32b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


-> Response received


Chat ended.


c:\Valinor\Programs\miniconda3\envs\llm_zoomcamp\Lib\site-packages\toyaikit\chat\runners.py:184: UnknownModelWarning: No pricing data for model 'qwen/qwen3-32b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(
